# 🧠 Multimodal Image Registration — A Hands-On Tutorial

> **Runtime:** Google Colab · T4 GPU  
> **Topics:** Classical registration (SimpleITK), Deep Learning registration (PyTorch/VoxelMorph-style), Mutual Information, Evaluation metrics

---

## What is Image Registration?

Image registration is the process of **aligning two or more images** into a common coordinate frame.  
When the images come from **different imaging modalities** (e.g. MRI ↔ CT, T1 ↔ T2 MRI, RGB ↔ depth), this is called **multimodal registration**.

```
  Moving Image (M)          Fixed Image (F)
  ┌──────────────┐          ┌──────────────┐
  │  CT scan     │  ──T──▶  │  MRI scan    │
  │  (source)    │          │  (target)    │
  └──────────────┘          └──────────────┘
         Find transform T : M∘T ≈ F
```

### Why is Multimodal Registration Hard?

| Monomodal | Multimodal |
|-----------|------------|
| Same scanner, same physics | Different acquisition physics |
| Similar intensity distributions | Completely different intensity relationships |
| MSE works fine | Need information-theoretic metrics (MI, NMI) |

### Roadmap

1. **Part 1** — Setup & synthetic data generation  
2. **Part 2** — Similarity metrics deep-dive  
3. **Part 3** — Classical registration with SimpleITK (Rigid → Affine → Deformable B-spline)  
4. **Part 4** — Deep Learning registration (VoxelMorph-style U-Net, GPU-accelerated)  
5. **Part 5** — Evaluation: Dice, NMI, TRE  
6. **Part 6** — Speed & Practical Comparison
7. **Part 7** — Tips for Real-World Multimodal Registration
8. **Part 8** — Reflection Questions


---
## Part 1 — Setup & Installation


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install SimpleITK -q
!pip install antspyx -q          # Advanced Normalization Tools (Python)
!pip install dipy -q             # Diffusion imaging in Python (extra metrics)
!pip install nilearn nibabel -q  # Neuroimaging datasets & I/O
print("✅ Installation complete")

In [ ]:
import os, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize

import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

from scipy.ndimage import gaussian_filter, map_coordinates
from scipy.stats import entropy
from skimage.draw import disk, ellipse
from skimage.transform import warp, AffineTransform

# ── GPU check ─────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🚀 GPU detected: {gpu_name}  ({gpu_mem:.1f} GB VRAM)")
else:
    print("⚠️  No GPU found — some cells will be slower. Set Runtime → Change runtime type → T4 GPU")

print(f"   PyTorch {torch.__version__}  |  SimpleITK {sitk.__version__}")

### 1.1 — Synthetic Multimodal Phantom

We generate a **2-D brain-like phantom** with:
- A "T1-like" modality (bright GM, dark CSF)
- A "T2-like" modality (same anatomy, completely different contrast)
- A known ground-truth deformation so we can evaluate accuracy


In [ ]:
def make_brain_phantom(size=256, seed=42):
    """
    Returns a dict with tissue-label map and two multimodal image volumes.
    Labels: 0=background, 1=CSF, 2=Grey Matter, 3=White Matter, 4=lesion
    """
    rng = np.random.default_rng(seed)
    cx, cy = size // 2, size // 2
    labels = np.zeros((size, size), dtype=np.uint8)

    # Skull boundary (ellipse)
    rr, cc = ellipse(cx, cy, int(size*0.46), int(size*0.42))
    labels[rr, cc] = 1  # CSF by default inside skull

    # White matter core
    rr, cc = ellipse(cx, cy, int(size*0.30), int(size*0.28))
    labels[rr, cc] = 3

    # Grey matter ring (overwrite WM edge)
    rr, cc = ellipse(cx, cy, int(size*0.38), int(size*0.35))
    mask_gm = np.zeros_like(labels, dtype=bool)
    mask_gm[rr, cc] = True
    rr2, cc2 = ellipse(cx, cy, int(size*0.30), int(size*0.28))
    mask_gm[rr2, cc2] = False
    labels[mask_gm] = 2

    # Ventricles (dark CSF lakes in WM)
    for (dr, dc, ar, ac) in [(-30, -15, 18, 9), (-30, 15, 18, 9)]:
        rr, cc = ellipse(cx+dr, cy+dc, ar, ac)
        labels[rr, cc] = 1

    # Small lesion
    rr, cc = disk((cx-20, cy+40), 12)
    labels[rr, cc] = 4

    # ── Contrast tables (mean ± std per tissue) ─────────────────────────────
    #         BG    CSF    GM    WM    lesion
    T1_mu  = [  0,   30,  180,  230,   120]
    T1_std = [  2,    8,   12,   10,    15]
    T2_mu  = [  0,  220,  150,   80,   200]   # contrast INVERTED vs T1
    T2_std = [  2,   12,   10,    8,    20]

    def apply_contrast(lbl, mus, stds):
        img = np.zeros_like(lbl, dtype=np.float32)
        for k in range(5):
            mask = lbl == k
            img[mask] = rng.normal(mus[k], stds[k], mask.sum())
        return np.clip(img, 0, 255)

    T1 = apply_contrast(labels, T1_mu, T1_std)
    T2 = apply_contrast(labels, T2_mu, T2_std)
    # Smooth slightly (simulate scanner PSF)
    T1 = gaussian_filter(T1, sigma=1.2).astype(np.float32)
    T2 = gaussian_filter(T2, sigma=1.0).astype(np.float32)

    return {'labels': labels, 'T1': T1, 'T2': T2}


phantom = make_brain_phantom(256)
labels, T1, T2 = phantom['labels'], phantom['T1'], phantom['T2']

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
cmap_label = plt.cm.get_cmap('tab10', 5)
axes[0].imshow(labels, cmap=cmap_label, vmin=0, vmax=4)
axes[0].set_title('Ground-Truth Labels\n(0=BG, 1=CSF, 2=GM, 3=WM, 4=Lesion)', fontsize=11)
axes[1].imshow(T1, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('T1-like Modality\n(WM bright, CSF dark)', fontsize=11)
axes[2].imshow(T2, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('T2-like Modality\n(CSF bright, WM dark)', fontsize=11)
for ax in axes:
    ax.axis('off')
plt.suptitle('Synthetic Brain Phantom — Same Anatomy, Different Contrast', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Phantom shape: {T1.shape} | T1 range [{T1.min():.1f}, {T1.max():.1f}] | "
      f"T2 range [{T2.min():.1f}, {T2.max():.1f}]")

In [ ]:
# ── Apply a known ground-truth misalignment to create the Moving image ────────
def apply_rigid_transform(img, angle_deg=8.0, tx=18, ty=-12):
    """Rotate + translate an image. Returns warped image."""
    tf = AffineTransform(rotation=np.deg2rad(angle_deg),
                         translation=(tx, ty))
    return warp(img, tf.inverse, order=1,
                mode='constant', cval=0.0,
                preserve_range=True).astype(np.float32)

GT_ANGLE = 8.0   # degrees
GT_TX    = 18    # pixels
GT_TY    = -12   # pixels

# Fixed = T1, Moving = T2 misaligned
fixed   = T1.copy()
moving  = apply_rigid_transform(T2, GT_ANGLE, GT_TX, GT_TY)
labels_moving = apply_rigid_transform(labels.astype(np.float32),
                                       GT_ANGLE, GT_TX, GT_TY)

# Checkerboard visualisation helper
def checkerboard(imgA, imgB, tiles=8):
    """Interleave two images in a checkerboard pattern."""
    H, W = imgA.shape
    th, tw = H // tiles, W // tiles
    out = imgA.copy()
    for i in range(tiles):
        for j in range(tiles):
            if (i + j) % 2 == 0:
                r0, r1 = i*th, min((i+1)*th, H)
                c0, c1 = j*tw, min((j+1)*tw, W)
                out[r0:r1, c0:c1] = imgB[r0:r1, c0:c1]
    return out

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].imshow(fixed,  cmap='gray'); axes[0].set_title('Fixed (T1)',  fontsize=12)
axes[1].imshow(moving, cmap='gray'); axes[1].set_title('Moving (T2, misaligned)',  fontsize=12)
cb_pre = checkerboard(fixed / fixed.max(), moving / moving.max())
axes[2].imshow(cb_pre, cmap='gray')
axes[2].set_title(f'Checkerboard (before reg)\nΔangle={GT_ANGLE}°, Δtx={GT_TX}px, Δty={GT_TY}px', fontsize=11)
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

---
## Part 2 — Similarity Metrics for Multimodal Registration

### 2.1 — Why not MSE?

In monomodal registration, minimizing pixel-wise **Mean Squared Error (MSE)** works because identical anatomy has similar intensities.  
In multimodal registration, the same anatomy produces completely different intensities → MSE is **anti-correlated with alignment**.

### 2.2 — Mutual Information (MI)

$$\text{MI}(F, M) = \sum_{f,m} p_{FM}(f,m) \log \frac{p_{FM}(f,m)}{p_F(f)\, p_M(m)}$$

MI measures how much knowing one image's intensity reduces uncertainty about the other.  
**Maximised when images are aligned** regardless of contrast differences.

**Normalized Mutual Information (NMI)** is more robust to image overlap:

$$\text{NMI}(F, M) = \frac{H(F) + H(M)}{H(F, M)}$$

where $H$ is Shannon entropy.


In [ ]:
def compute_mi(img1, img2, bins=64):
    """Mutual Information via joint histogram."""
    hist_2d, _, _ = np.histogram2d(
        img1.ravel(), img2.ravel(), bins=bins,
        range=[[img1.min(), img1.max()],[img2.min(), img2.max()]])
    pxy  = hist_2d / hist_2d.sum()
    px   = pxy.sum(axis=1, keepdims=True)
    py   = pxy.sum(axis=0, keepdims=True)
    mask = pxy > 0
    mi   = (pxy[mask] * np.log(pxy[mask] / (px * py + 1e-10)[mask])).sum()
    return float(mi)

def compute_nmi(img1, img2, bins=64):
    """Normalized Mutual Information = (H(F)+H(M)) / H(F,M)."""
    hist_2d, xedges, yedges = np.histogram2d(
        img1.ravel(), img2.ravel(), bins=bins)
    pxy = hist_2d / hist_2d.sum()
    px  = pxy.sum(axis=1)
    py  = pxy.sum(axis=0)
    Hx  = entropy(px + 1e-10)
    Hy  = entropy(py + 1e-10)
    Hxy = entropy(pxy.ravel() + 1e-10)
    return (Hx + Hy) / (Hxy + 1e-10)

def compute_mse(img1, img2):
    return float(np.mean((img1 - img2)**2))

# Show joint histogram & metric landscape over rotation angles
angles = np.linspace(-20, 20, 81)
mse_vals, mi_vals, nmi_vals = [], [], []

for ang in angles:
    m = apply_rigid_transform(T2, angle_deg=ang + GT_ANGLE,  # sweep around GT
                               tx=GT_TX, ty=GT_TY)
    mse_vals.append(compute_mse(fixed, m))
    mi_vals.append(compute_mi(fixed, m))
    nmi_vals.append(compute_nmi(fixed, m))

# Joint histogram at aligned vs misaligned
aligned_T2 = apply_rigid_transform(T2, angle_deg=0, tx=0, ty=0)  # perfect alignment

fig = plt.figure(figsize=(16, 9))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)

# Joint histograms
for col, (img2, title) in enumerate([(aligned_T2, 'Aligned T1↔T2'),
                                      (moving,     'Misaligned T1↔T2')]):
    ax = fig.add_subplot(gs[0, col])
    h, xe, ye = np.histogram2d(fixed.ravel(), img2.ravel(), bins=80)
    ax.imshow(np.log1p(h).T, origin='lower', aspect='auto', cmap='hot')
    ax.set_xlabel('Fixed (T1) intensity', fontsize=9)
    ax.set_ylabel('Moving (T2) intensity', fontsize=9)
    nmi_val = compute_nmi(fixed, img2)
    ax.set_title(f'{title}\nNMI = {nmi_val:.3f}', fontsize=10)

# Metric landscapes
ax_mse = fig.add_subplot(gs[0, 2:4])
ax_mse.plot(angles, mse_vals, 'r-', lw=2, label='MSE (↓ = better)')
ax_mse.axvline(0, color='k', linestyle='--', label='GT alignment (ang offset=0)')
ax_mse.set_xlabel('Additional rotation offset (°)'); ax_mse.set_ylabel('MSE')
ax_mse.set_title('MSE Landscape — note: NOT minimum at GT', fontsize=10)
ax_mse.legend(fontsize=8)

ax_mi = fig.add_subplot(gs[1, 0:2])
ax_mi.plot(angles, mi_vals,  'b-', lw=2, label='MI (↑ = better)')
ax_mi.axvline(0, color='k', linestyle='--', label='GT alignment')
ax_mi.set_xlabel('Additional rotation offset (°)'); ax_mi.set_ylabel('MI')
ax_mi.set_title('MI Landscape — peaks at GT ✓', fontsize=10)
ax_mi.legend(fontsize=8)

ax_nmi = fig.add_subplot(gs[1, 2:4])
ax_nmi.plot(angles, nmi_vals, 'g-', lw=2, label='NMI (↑ = better)')
ax_nmi.axvline(0, color='k', linestyle='--', label='GT alignment')
ax_nmi.set_xlabel('Additional rotation offset (°)'); ax_nmi.set_ylabel('NMI')
ax_nmi.set_title('NMI Landscape — peaks at GT, more stable ✓', fontsize=10)
ax_nmi.legend(fontsize=8)

plt.suptitle('Why MSE Fails for Multimodal Registration — MI/NMI is Required', fontsize=13, fontweight='bold')
plt.show()

print(f"NMI (aligned):    {compute_nmi(fixed, aligned_T2):.4f}")
print(f"NMI (misaligned): {compute_nmi(fixed, moving):.4f}")

---
## Part 3 — Classical Registration with SimpleITK

SimpleITK provides production-quality implementations of the ITK registration framework:

```
  ┌─────────────┐     ┌──────────────────┐     ┌──────────────────┐
  │  Transform  │     │  Metric          │     │  Optimizer       │
  │  ─────────  │     │  ─────────       │     │  ─────────       │
  │  Rigid      │     │  Mattes MI       │     │  Gradient Desc.  │
  │  Affine     │  +  │  Joint Hist. MI  │  +  │  L-BFGS          │
  │  B-Spline   │     │  NMI             │     │  Exhaustive      │
  │             │     │  Mean Squares    │     │  Evolutionary    │
  └─────────────┘     └──────────────────┘     └──────────────────┘
```

### Strategy: Multi-Resolution Pyramid

We always register coarse-to-fine to avoid local minima:
```
  Level 3 (1/8 res) ──→  Level 2 (1/4 res) ──→  Level 1 (1/2 res) ──→  Level 0 (full res)
       fast, global         refine                  refine                  final polish
```


In [ ]:
# ── SimpleITK conversion helpers ──────────────────────────────────────────────
def np_to_sitk(arr):
    """numpy float32 → SimpleITK Image (normalised to [0,1])."""
    a = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    return sitk.GetImageFromArray(a.astype(np.float32))

def sitk_to_np(img):
    return sitk.GetArrayFromImage(img)

def setup_registration(fixed_sitk, moving_sitk,
                        transform_type='rigid',
                        metric='mattes_mi',
                        n_bins=50,
                        shrink_factors=[4,2,1],
                        smooth_sigmas=[2,1,0]):
    """
    Build and configure an ImageRegistrationMethod.
    transform_type : 'rigid' | 'affine' | 'bspline'
    metric         : 'mattes_mi' | 'joint_hist_mi' | 'mean_squares' | 'correlation'
    """
    R = sitk.ImageRegistrationMethod()

    # ── Transform ──────────────────────────────────────────────────────────
    if transform_type == 'rigid':
        tx = sitk.CenteredTransformInitializer(
            fixed_sitk, moving_sitk,
            sitk.Euler2DTransform(),
            sitk.CenteredTransformInitializerFilter.GEOMETRY)
        R.SetInitialTransform(tx, inPlace=False)

    elif transform_type == 'affine':
        tx = sitk.CenteredTransformInitializer(
            fixed_sitk, moving_sitk,
            sitk.AffineTransform(2),
            sitk.CenteredTransformInitializerFilter.GEOMETRY)
        R.SetInitialTransform(tx, inPlace=False)

    elif transform_type == 'bspline':
        grid_spacing = [fixed_sitk.GetSize()[0] // 8] * 2   # 8×8 control point grid
        tx = sitk.BSplineTransformInitializer(
            image1=fixed_sitk,
            transformDomainMeshSize=[fixed_sitk.GetSize()[0]//grid_spacing[0]] * 2,
            order=3)
        R.SetInitialTransform(tx, inPlace=True)

    # ── Metric ─────────────────────────────────────────────────────────────
    if metric == 'mattes_mi':
        R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=n_bins)
        R.SetMetricSamplingStrategy(R.RANDOM)
        R.SetMetricSamplingPercentage(0.15)
    elif metric == 'joint_hist_mi':
        R.SetMetricAsJointHistogramMutualInformation(numberOfHistogramBins=n_bins)
    elif metric == 'mean_squares':
        R.SetMetricAsMeanSquares()
    elif metric == 'correlation':
        R.SetMetricAsCorrelation()

    # ── Optimizer ──────────────────────────────────────────────────────────
    if transform_type == 'bspline':
        R.SetOptimizerAsLBFGSB(gradientConvergenceTolerance=1e-5,
                                numberOfIterations=150,
                                maximumNumberOfCorrections=5,
                                maximumNumberOfFunctionEvaluations=200)
    else:
        R.SetOptimizerAsGradientDescentLineSearch(
            learningRate=1.0,
            numberOfIterations=200,
            convergenceMinimumValue=1e-6,
            convergenceWindowSize=10)
    R.SetOptimizerScalesFromPhysicalShift()

    # ── Multi-resolution ───────────────────────────────────────────────────
    R.SetShrinkFactorsPerLevel(shrinkFactors=shrink_factors)
    R.SetSmoothingSigmasPerLevel(smoothingSigmas=smooth_sigmas)
    R.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    R.SetInterpolator(sitk.sitkLinear)

    return R

print("✅ Registration utilities ready")

### 3.1 — Rigid Registration (Rotation + Translation)

6 DOF in 3D / 3 DOF in 2D. Best for brain MRI where anatomy is assumed rigid.

In [ ]:
fixed_sitk  = np_to_sitk(fixed)
moving_sitk = np_to_sitk(moving)

# Track metric values for plotting
metric_log = []
def metric_callback(R):
    metric_log.append(R.GetMetricValue())

R_rigid = setup_registration(fixed_sitk, moving_sitk,
                               transform_type='rigid',
                               metric='mattes_mi',
                               shrink_factors=[4,2,1],
                               smooth_sigmas=[2,1,0])
metric_log.clear()
R_rigid.AddCommand(sitk.sitkIterationEvent, lambda: metric_callback(R_rigid))

t0 = time.time()
tx_rigid = R_rigid.Execute(fixed_sitk, moving_sitk)
t_rigid  = time.time() - t0

registered_rigid = sitk_to_np(
    sitk.Resample(moving_sitk, fixed_sitk, tx_rigid,
                  sitk.sitkLinear, 0.0, moving_sitk.GetPixelID()))

# Extract estimated transform params
tx_params = tx_rigid.GetParameters()
est_angle = np.rad2deg(tx_params[0])
est_tx    = tx_params[1]
est_ty    = tx_params[2]

print(f"\n{'='*55}")
print(f"  RIGID REGISTRATION RESULTS")
print(f"{'='*55}")
print(f"  Elapsed time     : {t_rigid:.2f}s")
print(f"  Final metric     : {R_rigid.GetMetricValue():.5f}")
print(f"  Ground truth     : angle={GT_ANGLE:.1f}°, tx={GT_TX}, ty={GT_TY}")
print(f"  Estimated        : angle={est_angle:.2f}°, tx={est_tx:.2f}, ty={est_ty:.2f}")
print(f"  Angle error      : {abs(est_angle - GT_ANGLE):.2f}°")
print(f"{'='*55}")

# Visualisation
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(fixed,            cmap='gray'); axes[0].set_title('Fixed (T1)')
axes[1].imshow(moving,           cmap='gray'); axes[1].set_title('Moving (T2, misaligned)')
axes[2].imshow(registered_rigid, cmap='gray'); axes[2].set_title(f'Rigid Registered')
cb_post = checkerboard(fixed/fixed.max(), registered_rigid/registered_rigid.max())
axes[3].imshow(cb_post, cmap='gray');          axes[3].set_title('Checkerboard (after rigid)')
for ax in axes: ax.axis('off')

# Overlay metric convergence
fig2, ax2 = plt.subplots(figsize=(8, 3))
ax2.plot(metric_log, 'b-', lw=1.5)
ax2.set_xlabel('Iteration'); ax2.set_ylabel('Mattes MI (negative)')
ax2.set_title('Optimizer Convergence — Rigid Registration')
plt.tight_layout()
plt.show()

### 3.2 — Affine Registration

Adds scaling and shearing to rigid: 12 DOF in 3D / 6 DOF in 2D.  
Good for multi-site scanner differences, field distortions, or inter-subject registration.

In [ ]:
# Apply an affine misalignment (rotation + shear + scale)
from skimage.transform import AffineTransform

tf_affine = AffineTransform(rotation=np.deg2rad(6),
                             shear=0.05,
                             scale=(1.03, 0.97),
                             translation=(10, -8))
moving_affine = warp(T2, tf_affine.inverse, order=1,
                      mode='constant', cval=0.0,
                      preserve_range=True).astype(np.float32)

moving_affine_sitk = np_to_sitk(moving_affine)

metric_log_affine = []
R_affine = setup_registration(fixed_sitk, moving_affine_sitk,
                               transform_type='affine',
                               metric='mattes_mi',
                               shrink_factors=[4,2,1],
                               smooth_sigmas=[2,1,0])
R_affine.AddCommand(sitk.sitkIterationEvent,
                    lambda: metric_log_affine.append(R_affine.GetMetricValue()))

t0 = time.time()
tx_affine = R_affine.Execute(fixed_sitk, moving_affine_sitk)
t_affine  = time.time() - t0

registered_affine = sitk_to_np(
    sitk.Resample(moving_affine_sitk, fixed_sitk, tx_affine,
                  sitk.sitkLinear, 0.0, moving_affine_sitk.GetPixelID()))

print(f"Affine registration completed in {t_affine:.2f}s")
print(f"Final metric value: {R_affine.GetMetricValue():.5f}")
print(f"NMI before: {compute_nmi(fixed, moving_affine):.4f}")
print(f"NMI after : {compute_nmi(fixed, registered_affine):.4f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(fixed,              cmap='gray'); axes[0].set_title('Fixed (T1)')
axes[1].imshow(moving_affine,      cmap='gray'); axes[1].set_title('Moving (T2 + shear)')
axes[2].imshow(registered_affine,  cmap='gray'); axes[2].set_title('Affine Registered')
cb_aff = checkerboard(fixed/fixed.max(), registered_affine/registered_affine.max())
axes[3].imshow(cb_aff, cmap='gray'); axes[3].set_title('Checkerboard (after affine)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

### 3.3 — Deformable B-Spline Registration

For non-rigid anatomy differences (inter-subject, tumour deformation, breathing motion).  
B-splines offer smooth, locally-controlled displacement fields via a sparse control point grid:

$$\mathbf{u}(\mathbf{x}) = \sum_{i,j} \mathbf{c}_{ij} \, B_i(x) B_j(y)$$

⚠️ **Always initialise with rigid/affine first**, then refine with B-spline.


In [ ]:
# Add a local non-rigid deformation on top of the rigid misalignment
def apply_local_deformation(img, strength=8.0, seed=0):
    """Smooth random displacement field applied to image."""
    rng = np.random.default_rng(seed)
    H, W = img.shape
    ux = gaussian_filter(rng.normal(0, 1, (H, W)), sigma=25) * strength
    uy = gaussian_filter(rng.normal(0, 1, (H, W)), sigma=25) * strength
    coords_r = np.arange(H)[:, None] + ux
    coords_c = np.arange(W)[None, :] + uy
    return map_coordinates(img, [coords_r, coords_c],
                           order=1, mode='constant').astype(np.float32)

# Apply rigid + local deformation
moving_deform = apply_rigid_transform(
    apply_local_deformation(T2, strength=10), angle_deg=5, tx=12, ty=-8)

moving_deform_sitk = np_to_sitk(moving_deform)

# Step 1: Rigid init
R_init = setup_registration(fixed_sitk, moving_deform_sitk,
                              transform_type='rigid', metric='mattes_mi',
                              shrink_factors=[4,2,1], smooth_sigmas=[2,1,0])
tx_init = R_init.Execute(fixed_sitk, moving_deform_sitk)

# Apply rigid init
moving_init = sitk.Resample(moving_deform_sitk, fixed_sitk, tx_init,
                             sitk.sitkLinear, 0.0, moving_deform_sitk.GetPixelID())

# Step 2: B-spline refinement
metric_log_bspline = []
R_bsp = setup_registration(fixed_sitk, moving_init,
                             transform_type='bspline', metric='mattes_mi',
                             shrink_factors=[2,1], smooth_sigmas=[1,0])
R_bsp.AddCommand(sitk.sitkIterationEvent,
                 lambda: metric_log_bspline.append(R_bsp.GetMetricValue()))

t0 = time.time()
tx_bsp = R_bsp.Execute(fixed_sitk, moving_init)
t_bsp  = time.time() - t0

registered_bspline = sitk_to_np(
    sitk.Resample(moving_init, fixed_sitk, tx_bsp,
                  sitk.sitkLinear, 0.0, moving_init.GetPixelID()))

print(f"B-Spline refinement completed in {t_bsp:.2f}s")
print(f"NMI before: {compute_nmi(fixed, moving_deform):.4f}")
print(f"NMI after rigid: {compute_nmi(fixed, sitk_to_np(moving_init)):.4f}")
print(f"NMI after bspline: {compute_nmi(fixed, registered_bspline):.4f}")

# Visualise displacement field magnitude
disp_field = sitk.TransformToDisplacementField(
    tx_bsp, sitk.sitkVectorFloat64, fixed_sitk.GetSize(),
    fixed_sitk.GetOrigin(), fixed_sitk.GetSpacing(), fixed_sitk.GetDirection())
disp_np = sitk.GetArrayFromImage(disp_field)   # shape (H,W,2)
disp_mag = np.sqrt((disp_np**2).sum(axis=-1))

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
axes[0].imshow(fixed,               cmap='gray'); axes[0].set_title('Fixed (T1)')
axes[1].imshow(moving_deform,       cmap='gray'); axes[1].set_title('Moving (deformed T2)')
axes[2].imshow(registered_bspline,  cmap='gray'); axes[2].set_title('B-Spline Registered')
im = axes[3].imshow(disp_mag, cmap='jet')
axes[3].set_title('Displacement Field Magnitude')
plt.colorbar(im, ax=axes[3], fraction=0.046)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---
## Part 4 — Deep Learning Registration with a VoxelMorph-style U-Net (T4 GPU)

Classical registration is slow (iterative optimisation per image pair). Deep learning learns a **registration function** once and runs in milliseconds at inference.

### Architecture — Symmetric U-Net Encoder-Decoder

```
  [Fixed | Moving] ──→ Encoder ──→ Bottleneck ──→ Decoder ──→ Displacement Field u(x)
  (2 channels)          skip connections                        ↓
                                                          Spatial Transformer
                                                                ↓
                                                        Moved Image M∘u
```

### Loss Function

$$\mathcal{L} = \underbrace{\mathcal{L}_{\text{sim}}(F, M\circ u)}_{\text{image similarity}} + \lambda \underbrace{\|\nabla u\|^2}_{\text{smoothness regulariser}}$$

For multimodal, we use **Local Normalised Cross-Correlation (LNCC)** as $\mathcal{L}_{\text{sim}}$ — it is differentiable and handles contrast differences locally.

In [ ]:
# ── Spatial Transformer Network ───────────────────────────────────────────────
class SpatialTransformer(nn.Module):
    """
    Differentiable bilinear warping using a dense displacement field.
    field : (B, 2, H, W) displacement field in pixels
    """
    def __init__(self, size):
        super().__init__()
        H, W = size
        # Build identity sampling grid
        vectors = [torch.arange(0, s) for s in [H, W]]
        grids   = torch.meshgrid(vectors, indexing='ij')
        grid    = torch.stack(grids)  # (2, H, W)
        grid    = torch.unsqueeze(grid, 0).float()  # (1,2,H,W)
        self.register_buffer('grid', grid)

    def forward(self, src, flow):
        """Warp src image with displacement flow."""
        H, W = src.shape[-2:]
        new_locs = self.grid + flow  # add displacement to identity grid
        # Normalise to [-1, 1] for grid_sample
        new_locs[:, 0, ...] = 2.0 * new_locs[:, 0, ...] / max(H - 1, 1) - 1.0
        new_locs[:, 1, ...] = 2.0 * new_locs[:, 1, ...] / max(W - 1, 1) - 1.0
        # grid_sample expects (B,H,W,2) with (x,y) order
        grid_t = new_locs.permute(0, 2, 3, 1)[..., [1, 0]]
        return F.grid_sample(src, grid_t,
                             align_corners=True, mode='bilinear',
                             padding_mode='zeros')


# ── U-Net Registration Network ────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x): return self.block(x)


class VoxelMorphUNet(nn.Module):
    """
    Lightweight 2-D VoxelMorph-style registration network.
    Input : (B, 2, H, W) — concatenated [fixed, moving]
    Output: (B, 2, H, W) — dense displacement field
    """
    def __init__(self, img_size=(256, 256), base_ch=16):
        super().__init__()
        c = base_ch
        # Encoder
        self.enc1 = ConvBlock(2,   c,   stride=1)  # 256→256
        self.enc2 = ConvBlock(c,   c*2, stride=2)  # 256→128
        self.enc3 = ConvBlock(c*2, c*4, stride=2)  # 128→ 64
        self.enc4 = ConvBlock(c*4, c*8, stride=2)  #  64→ 32
        # Bottleneck
        self.bot  = ConvBlock(c*8, c*8, stride=2)  #  32→ 16
        # Decoder (with skip connections)
        self.up4  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec4 = ConvBlock(c*8 + c*8, c*8)
        self.up3  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec3 = ConvBlock(c*8 + c*4, c*4)
        self.up2  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec2 = ConvBlock(c*4 + c*2, c*2)
        self.up1  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec1 = ConvBlock(c*2 + c,   c)
        # Flow head — small init to start near identity
        self.flow = nn.Conv2d(c, 2, 3, padding=1)
        nn.init.normal_(self.flow.weight, mean=0, std=1e-5)
        nn.init.zeros_(self.flow.bias)
        # Spatial transformer
        self.stn  = SpatialTransformer(img_size)

    def forward(self, fixed, moving):
        x  = torch.cat([fixed, moving], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b  = self.bot(e4)
        d4 = self.dec4(torch.cat([self.up4(b),  e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        flow = self.flow(d1)
        moved = self.stn(moving, flow)
        return moved, flow


model = VoxelMorphUNet(img_size=(256, 256), base_ch=16).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"VoxelMorphUNet: {n_params:,} parameters  |  device: {device}")

### Loss Function

$$\mathcal{L} = \underbrace{\mathcal{L}_{\text{sim}}(F, M\circ u)}_{\text{image similarity}} + \lambda \underbrace{\|\nabla u\|^2}_{\text{smoothness regulariser}}$$

In [ ]:
# ── Loss functions ────────────────────────────────────────────────────────────

class LocalNCC(nn.Module):
    """
    Local Normalized Cross-Correlation — differentiable proxy for NMI.
    Works across modalities because it measures local correlation, not intensity equality.
    win : half-window size (total window = 2*win+1)
    """
    def __init__(self, win=9):
        super().__init__()
        self.win = win
        self.kernel_size = 2 * win + 1
        # Box filter kernel (averages in local windows)
        kernel = torch.ones(1, 1, self.kernel_size, self.kernel_size)
        self.register_buffer('kernel', kernel / kernel.sum())

    def forward(self, fixed, moving):
        pad = self.win
        def local_mean(x):
            return F.conv2d(x, self.kernel, padding=pad)
        Ef  = local_mean(fixed)
        Em  = local_mean(moving)
        Eff = local_mean(fixed  * fixed)  - Ef * Ef
        Emm = local_mean(moving * moving) - Em * Em
        Efm = local_mean(fixed  * moving) - Ef * Em
        ncc = (Efm * Efm) / (Eff * Emm + 1e-5)
        return -ncc.mean()   # we minimise, so negate


class GradientSmoothnessLoss(nn.Module):
    """Penalise spatial derivatives of the displacement field (Tikhonov regularisation)."""
    def forward(self, flow):
        dy = (flow[:, :, 1:, :] - flow[:, :, :-1, :]).pow(2)
        dx = (flow[:, :, :, 1:] - flow[:, :, :, :-1]).pow(2)
        return (dy.mean() + dx.mean()) / 2


loss_sim    = LocalNCC(win=9).to(device)
loss_smooth = GradientSmoothnessLoss().to(device)
print("✅ LNCC + Gradient Smoothness losses ready")

In [ ]:
# ── Synthetic dataset generator for training ──────────────────────────────────
class MultimodalPairDataset(torch.utils.data.Dataset):
    """
    Generates batches of (fixed=T1, moving=T2_deformed) pairs on the fly.
    Each call produces a new random deformation for data augmentation.
    """
    def __init__(self, T1, T2, n_samples=400, size=256, deform_strength=12):
        self.T1 = T1.astype(np.float32)
        self.T2 = T2.astype(np.float32)
        self.n  = n_samples
        self.sz = size
        self.ds = deform_strength
        # Normalise to [0,1]
        self.T1 = (self.T1 - self.T1.min()) / (self.T1.max() - self.T1.min())
        self.T2 = (self.T2 - self.T2.min()) / (self.T2.max() - self.T2.min())

    def __len__(self): return self.n

    def __getitem__(self, idx):
        rng = np.random.default_rng(idx)
        angle = rng.uniform(-15, 15)
        tx    = rng.integers(-20, 20)
        ty    = rng.integers(-20, 20)
        # Rigid part
        tf = AffineTransform(rotation=np.deg2rad(angle),
                             translation=(tx, ty))
        T2_warp = warp(self.T2, tf.inverse, order=1,
                       preserve_range=True, mode='constant').astype(np.float32)
        # Non-rigid part
        H, W = self.T2.shape
        ux = gaussian_filter(rng.normal(0,1,(H,W)), sigma=30) * self.ds
        uy = gaussian_filter(rng.normal(0,1,(H,W)), sigma=30) * self.ds
        cr = np.arange(H)[:,None] + ux
        cc = np.arange(W)[None,:] + uy
        T2_warp = map_coordinates(T2_warp, [cr, cc],
                                  order=1, mode='constant').astype(np.float32)
        fixed  = torch.from_numpy(self.T1)[None]    # (1,H,W)
        moving = torch.from_numpy(T2_warp)[None]    # (1,H,W)
        return fixed, moving


dataset    = MultimodalPairDataset(T1, T2, n_samples=400)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=4,
                                          shuffle=True, num_workers=2)
print(f"Dataset: {len(dataset)} samples  |  Batch size: 4  |  Steps/epoch: {len(dataloader)}")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
LAMBDA_SMOOTH = 1.5   # regularisation weight
N_EPOCHS      = 25    # ≈ 2–3 min on T4
LR            = 1e-4

optimizer = Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

train_losses, sim_losses, reg_losses = [], [], []

print(f"Training for {N_EPOCHS} epochs on {device}...")
print(f"{'Epoch':>6} {'Total':>10} {'LNCC':>10} {'Smooth':>10} {'LR':>12} {'Time':>8}")
print("-" * 60)

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    ep_total, ep_sim, ep_reg = 0., 0., 0.
    t_ep = time.time()

    for fixed_b, moving_b in dataloader:
        fixed_b  = fixed_b.to(device)
        moving_b = moving_b.to(device)

        moved, flow = model(fixed_b, moving_b)

        l_sim = loss_sim(fixed_b, moved)
        l_reg = loss_smooth(flow)
        loss  = l_sim + LAMBDA_SMOOTH * l_reg

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        ep_total += loss.item()
        ep_sim   += l_sim.item()
        ep_reg   += l_reg.item()

    scheduler.step()
    n = len(dataloader)
    train_losses.append(ep_total / n)
    sim_losses.append(ep_sim / n)
    reg_losses.append(ep_reg / n)
    lr_now = scheduler.get_last_lr()[0]

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>6} {ep_total/n:>10.4f} {ep_sim/n:>10.4f} "
              f"{ep_reg/n:>10.4f} {lr_now:>12.2e} {time.time()-t_ep:>7.1f}s")

print("\n✅ Training complete")

In [ ]:
# ── Plot training curves & inference example ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, len(train_losses)+1)
ax1.plot(epochs, train_losses, 'k-',  lw=2, label='Total')
ax1.plot(epochs, sim_losses,   'b--', lw=2, label='LNCC (sim)')
ax1.plot(epochs, reg_losses,   'r:',  lw=2, label='Smoothness (reg)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Curves'); ax1.legend()

# Inference on the original misaligned pair
model.eval()
with torch.no_grad():
    f_norm = (fixed - fixed.min())  / (fixed.max()  - fixed.min()  + 1e-8)
    m_norm = (moving - moving.min()) / (moving.max() - moving.min() + 1e-8)
    f_t = torch.from_numpy(f_norm[None,None]).to(device)
    m_t = torch.from_numpy(m_norm[None,None]).to(device)
    t0_inf = time.time()
    moved_dl, flow_dl = model(f_t, m_t)
    t_inf = (time.time() - t0_inf) * 1000

moved_dl_np = moved_dl[0,0].cpu().numpy()
flow_dl_np  = flow_dl[0].cpu().numpy()   # (2,H,W)
disp_mag_dl = np.sqrt((flow_dl_np**2).sum(axis=0))

# Quiver subsample
step = 16
Y, X = np.mgrid[0:256:step, 0:256:step]
U = flow_dl_np[0, ::step, ::step]
V = flow_dl_np[1, ::step, ::step]

ax2.imshow(disp_mag_dl, cmap='plasma')
ax2.quiver(X, Y, V, U, color='white', alpha=0.7, scale=200, width=0.003)
ax2.set_title(f'DL Displacement Field\n(inference: {t_inf:.1f} ms on {device})')
ax2.axis('off')
plt.tight_layout(); plt.show()

# Full comparison
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(fixed,       cmap='gray'); axes[0].set_title('Fixed (T1)')
axes[1].imshow(moving,      cmap='gray'); axes[1].set_title('Moving (T2, misaligned)')
axes[2].imshow(moved_dl_np, cmap='gray'); axes[2].set_title('DL Registered')
cb_dl = checkerboard(f_norm, moved_dl_np)
axes[3].imshow(cb_dl, cmap='gray');       axes[3].set_title('Checkerboard (after DL reg)')
for ax in axes: ax.axis('off')
plt.suptitle('Deep Learning Registration Result', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"NMI before: {compute_nmi(fixed, moving):.4f}")
print(f"NMI after DL: {compute_nmi(f_norm, moved_dl_np):.4f}")
print(f"Inference time: {t_inf:.1f} ms")

---
## Part 5 — Evaluation Metrics

| Metric | Measures | Requires |
|--------|----------|----------|
| **NMI** | Intensity alignment | Nothing extra |
| **Dice Score** | Anatomical overlap | Segmentation masks |
| **TRE** | Landmark distance | Corresponding landmarks |
| **Jacobian det.** | Deformation plausibility | Displacement field |
| **Hausdorff dist.** | Surface error | Segmentation contours |


Dice Score: A score of 1 means perfect overlap, 0 means no overlap at all.

TRE: Target Registration Error.

In [ ]:
# ── Evaluation utilities ──────────────────────────────────────────────────────

def dice_score(seg_a, seg_b, labels_list=None):
    """Compute per-label Dice coefficient between two label maps."""
    if labels_list is None:
        labels_list = np.unique(seg_a)[1:]   # skip background
    scores = {}
    for lbl in labels_list:
        A = seg_a == lbl
        B = seg_b == lbl
        intersection = (A & B).sum()
        dice = 2 * intersection / (A.sum() + B.sum() + 1e-8)
        scores[int(lbl)] = float(dice)
    return scores

def target_registration_error(lm_fixed, lm_moved):
    """
    Mean Euclidean distance between corresponding landmark sets.
    lm_fixed, lm_moved: (N,2) arrays of (row, col) pixel coordinates
    """
    return float(np.mean(np.linalg.norm(lm_fixed - lm_moved, axis=1)))

def jacobian_determinant_stats(flow):
    """
    Compute Jacobian determinant of displacement field.
    Negative det = folding (non-diffeomorphic) ← bad!
    flow: (2, H, W) displacement field
    """
    dy_ux = flow[0, 1:, :] - flow[0, :-1, :]
    dx_ux = flow[0, :, 1:] - flow[0, :, :-1]
    dy_uy = flow[1, 1:, :] - flow[1, :-1, :]
    dx_uy = flow[1, :, 1:] - flow[1, :, :-1]
    # Central crop for valid region
    J = (1 + dy_ux[:, :-1]) * (1 + dx_uy[:-1, :]) - dx_ux[:-1, :] * dy_uy[:, :-1]
    pct_folding = float((J < 0).mean() * 100)
    return {'mean': float(J.mean()), 'std': float(J.std()),
            'min': float(J.min()), 'max': float(J.max()),
            'pct_negative': pct_folding}


# ── Run full evaluation pipeline ──────────────────────────────────────────────
# Warp the moving label map with each method

# Ground truth: T2 labels (before any misalignment)
gt_labels = labels.copy()

# After rigid registration — warp label map with rigid tx
labels_sitk = sitk.GetImageFromArray(labels_moving.astype(np.float32))
labels_rigid_reg = np.round(sitk_to_np(
    sitk.Resample(labels_sitk, fixed_sitk, tx_rigid,
                  sitk.sitkNearestNeighbor, 0.0,
                  labels_sitk.GetPixelID()))).astype(np.uint8)

# After DL registration — warp label map using DL flow
lm_np = labels_moving.astype(np.float32)
lm_np = (lm_np - lm_np.min()) / (lm_np.max() - lm_np.min() + 1e-8)
lm_t  = torch.from_numpy(lm_np[None,None]).to(device)
with torch.no_grad():
    labels_dl_reg, _ = model(f_t, lm_t)
labels_dl_reg = np.round(
    labels_dl_reg[0,0].cpu().numpy() * 4).clip(0,4).astype(np.uint8)

# Dice
LABEL_NAMES = {1: 'CSF', 2: 'Grey Matter', 3: 'White Matter', 4: 'Lesion'}
dice_before = dice_score(gt_labels, labels_moving.astype(np.uint8))
dice_rigid  = dice_score(gt_labels, labels_rigid_reg)
dice_dl     = dice_score(gt_labels, labels_dl_reg)

# Jacobian stats (DL flow)
jac_stats = jacobian_determinant_stats(flow_dl_np)

# Summary table
print(f"\n{'='*65}")
print(f"  EVALUATION SUMMARY")
print(f"{'='*65}")
print(f"  {'Label':<20} {'Before':>10} {'Rigid':>10} {'DL':>10}")
print(f"  {'-'*52}")
for lbl, name in LABEL_NAMES.items():
    print(f"  {f'Dice {name}':<20} "
          f"{dice_before.get(lbl,0):>10.3f} "
          f"{dice_rigid.get(lbl,0):>10.3f} "
          f"{dice_dl.get(lbl,0):>10.3f}")
print(f"  {'-'*52}")
mean_bef = np.mean(list(dice_before.values()))
mean_rig = np.mean(list(dice_rigid.values()))
mean_dl  = np.mean(list(dice_dl.values()))
print(f"  {'Mean Dice':<20} {mean_bef:>10.3f} {mean_rig:>10.3f} {mean_dl:>10.3f}")
print(f"  {'-'*52}")
print(f"  {'NMI':<20} {compute_nmi(fixed,moving):>10.4f} "
      f"{compute_nmi(fixed,registered_rigid):>10.4f} "
      f"{compute_nmi(f_norm,moved_dl_np):>10.4f}")
print(f"{'='*65}")
print(f"\n  DL Jacobian determinant stats:")
for k,v in jac_stats.items():
    print(f"    {k:20s}: {v:.4f}")
print(f"  {'(pct_negative = 0 is ideal; >1% indicates folding)'}")

In [ ]:
# Visualise evaluation as bar chart
labels_names = [LABEL_NAMES[k] for k in sorted(LABEL_NAMES)]
x = np.arange(len(labels_names))
w = 0.28

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Dice bars
b_vals = [dice_before.get(k,0) for k in sorted(LABEL_NAMES)]
r_vals = [dice_rigid.get(k,0)  for k in sorted(LABEL_NAMES)]
d_vals = [dice_dl.get(k,0)     for k in sorted(LABEL_NAMES)]
ax1.bar(x - w,   b_vals, w, label='Before reg',    color='#e74c3c', alpha=0.85)
ax1.bar(x,       r_vals, w, label='Rigid (ITK)',   color='#3498db', alpha=0.85)
ax1.bar(x + w,   d_vals, w, label='DL (VoxelMorph)',color='#2ecc71', alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(labels_names, rotation=10)
ax1.set_ylabel('Dice Score'); ax1.set_ylim(0, 1)
ax1.set_title('Dice Score by Tissue Class', fontsize=12)
ax1.legend(); ax1.grid(axis='y', alpha=0.3)

# NMI / mean Dice comparison
methods = ['Before', 'Rigid', 'DL']
nmi_vals2 = [compute_nmi(fixed,moving),
             compute_nmi(fixed,registered_rigid),
             compute_nmi(f_norm,moved_dl_np)]
dice_avg = [mean_bef, mean_rig, mean_dl]
colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = ax2.bar(methods, nmi_vals2, color=colors, alpha=0.85, width=0.5)
ax2b = ax2.twinx()
ax2b.plot(methods, dice_avg, 'ko-', lw=2, ms=8, label='Mean Dice (right)')
ax2.set_ylabel('NMI'); ax2b.set_ylabel('Mean Dice')
ax2.set_title('NMI and Mean Dice Comparison', fontsize=12)
ax2b.set_ylim(0, 1)
ax2.legend(['NMI (left)'], loc='upper left')
ax2b.legend(loc='lower right')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Registration Quality Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Part 6 — Speed & Practical Comparison

One of the biggest advantages of deep learning registration is inference speed.


In [ ]:
# ── Benchmark: classical vs DL inference time ─────────────────────────────────
import timeit

N_REPEATS = 10

# Time classical rigid registration
def run_rigid():
    R = setup_registration(fixed_sitk, moving_sitk,
                            transform_type='rigid', metric='mattes_mi',
                            shrink_factors=[2,1], smooth_sigmas=[1,0])
    R.Execute(fixed_sitk, moving_sitk)

t_classical = timeit.timeit(run_rigid, number=N_REPEATS) / N_REPEATS * 1000

# Time DL inference
model.eval()
torch.cuda.synchronize() if torch.cuda.is_available() else None
def run_dl():
    with torch.no_grad():
        _ = model(f_t, m_t)
    torch.cuda.synchronize() if torch.cuda.is_available() else None

run_dl()  # warmup
t_dl = timeit.timeit(run_dl, number=N_REPEATS) / N_REPEATS * 1000

# Compute speedup
speedup = t_classical / t_dl

print(f"\n{'='*55}")
print(f"  SPEED COMPARISON (averaged over {N_REPEATS} runs)")
print(f"{'='*55}")
print(f"  Classical (SimpleITK rigid, {device}): {t_classical:>8.1f} ms")
print(f"  Deep Learning ({device}):             {t_dl:>8.1f} ms")
print(f"  Speedup:                              {speedup:>8.1f}×")
print(f"{'='*55}")

# Summary comparison table
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
table_data = [
    ['Method', 'Inference Time', 'Requires Training', 'Generalises', 'Deformable'],
    ['SimpleITK Rigid',   f'{t_classical:.0f} ms/pair', 'No',  'N/A',         'No'],
    ['SimpleITK B-Spline','~5–30 s/pair',               'No',  'N/A',         'Yes'],
    ['VoxelMorph (DL)',   f'{t_dl:.0f} ms/pair',        'Yes (once)', 'Yes',   'Yes'],
]
colors_row = [['#2c3e50']*5] + [['#ecf0f1']*5, ['#d5e8d4']*5, ['#dae8fc']*5, ]
font_colors = [['white']*5] + [['black']*5]*4
tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0],
               cellLoc='center', loc='center',
               cellColours=colors_row[1:],
               colColours=colors_row[0])
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 2.2)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_text_props(color='white', fontweight='bold')
ax.set_title('Method Comparison Summary', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout(); plt.show()

---
## Part 7 — Tips for Real-World Multimodal Registration

### ✅ Best Practices Checklist

**Data Preprocessing**
- Always **skull-strip** brain MRI before registration (removes confounding scalp tissue)
- **Intensity normalise** each image independently (`z-score`, `min-max`, or `histogram equalisation`)
- Resample to **isotropic voxel spacing** before registration
- Check image **orientation/affine headers** match (RAS vs LAS etc.)

**Classical Registration**
- Always use **multi-resolution** pyramids (prevents local minima)
- **Rigid before Affine before Deformable** — never start with B-spline
- Use `MattesMutualInformation` for CT↔MRI; `NMI` for T1↔T2 (more stable)
- Sample 10–20% of voxels randomly per iteration (faster, equally accurate)

**Deep Learning Registration**
- Training data should include the **full range of expected misalignments**
- Use **diffeomorphic integration** (Euler integration of SVF) to guarantee invertibility
- Tune `λ` smoothness weight: too low → unrealistic folds, too high → under-registration
- Consider **symmetric registration** (normalise by √(forward∘backward))

**Evaluation**
- Never report only NMI — it can improve while anatomy misaligns
- Always evaluate **Dice on held-out segmentations** for anatomical accuracy
- Check **Jacobian determinant** histogram: no/few negative values = physically valid
- For clinical use: **expert visual inspection** remains essential


In [ ]:
# ── Final comprehensive visualisation ─────────────────────────────────────────
fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(3, 5, figure=fig, hspace=0.40, wspace=0.05)

rows = [
    (fixed,             'Fixed\n(T1 MRI)'),
    (moving,            'Moving\n(T2 MRI, misaligned)'),
    (registered_rigid,  'Rigid Reg.\n(SimpleITK + MI)'),
    (registered_bspline,'B-Spline Reg.\n(Deformable)'),
    (moved_dl_np,       'DL Reg.\n(VoxelMorph, GPU)'),
]

seg_rows = [
    gt_labels,
    labels_moving.astype(np.uint8),
    labels_rigid_reg,
    labels_rigid_reg,       # placeholder (no separate bspline label warp here)
    labels_dl_reg,
]

for col, ((img, title), seg) in enumerate(zip(rows, seg_rows)):
    # Row 0: intensity images
    ax0 = fig.add_subplot(gs[0, col])
    ax0.imshow(img, cmap='gray', vmin=0)
    ax0.set_title(title, fontsize=9, fontweight='bold')
    ax0.axis('off')

    # Row 1: segmentation overlays
    ax1 = fig.add_subplot(gs[1, col])
    ax1.imshow(img, cmap='gray', vmin=0, alpha=0.7)
    ax1.imshow(np.ma.masked_equal(seg, 0),
               cmap=plt.cm.get_cmap('tab10', 5),
               vmin=0, vmax=4, alpha=0.5)
    ax1.axis('off')
    if col == 0:
        ax1.set_ylabel('Segmentation\nOverlay', fontsize=9)

    # Row 2: difference maps to fixed
    ax2 = fig.add_subplot(gs[2, col])
    img_norm = img / (img.max() + 1e-8)
    f_norm2  = fixed / (fixed.max() + 1e-8)
    diff = np.abs(img_norm - f_norm2)
    im2  = ax2.imshow(diff, cmap='hot', vmin=0, vmax=0.4)
    mse  = (diff**2).mean()
    ax2.set_title(f'|Δ| to Fixed\nMSE={mse:.4f}', fontsize=8)
    ax2.axis('off')

plt.suptitle('Complete Multimodal Registration Pipeline — Summary',
             fontsize=14, fontweight='bold', y=0.99)

# Shared colourbar for difference
fig.colorbar(im2, ax=fig.axes, shrink=0.25, pad=0.01, label='|Intensity diff|')
plt.show()

print("\n🎉 Tutorial complete!")
print("\nNext steps:")
print("  • Apply to real brain MRI: nilearn.datasets.fetch_icbm152_2009()")
print("  • Try 3D registration: SimpleITK works identically in 3D")
print("  • Implement diffeomorphic DL: integrate velocity field with scaling-and-squaring")
print("  • Explore ANTs SyN: the gold standard for neuroimaging registration")

---
## 📚 Further Reading

### Foundational Papers
- **Mattes et al. (2003)** — PET-CT registration using Mutual Information
- **Rueckert et al. (1999)** — Non-rigid B-spline registration of breast MRI
- **Balakrishnan et al. (2019)** — *VoxelMorph: A Learning Framework for Deformable Medical Image Registration*, IEEE TMI
- **Avants et al. (2008)** — *Symmetric Diffeomorphic Image Registration with Cross-Correlation (ANTs SyN)*, Medical Image Analysis

### Libraries
- [SimpleITK Documentation](https://simpleitk.readthedocs.io/) — includes 70+ Jupyter notebook examples
- [VoxelMorph GitHub](https://github.com/voxelmorph/voxelmorph) — official DL registration framework
- [ANTsPy](https://github.com/ANTsX/ANTsPy) — Python interface to ANTs
- [DIPY](https://dipy.org/) — diffeomorphic registration + diffusion imaging

### Datasets
| Dataset | Modalities | Size | Link |
|---------|-----------|------|------|
| IXI | T1/T2/PD MRI | 600 subjects | brain-development.org |
| OASIS | T1 MRI, longitudinal | 416 subjects | oasis-brains.org |
| Learn2Reg | CT/MRI, multi-organ | Challenge data | learn2reg.grand-challenge.org |
| LPBA40 | T1 + 54 manual segmentations | 40 subjects | loni.usc.edu |


---
## 🤔 Part 8 — Reflection Questions

Work through the questions below after completing the tutorial. They are grouped by theme and progress from recall to analysis to open-ended design.

---

### 📐 Fundamentals of Multimodal Registration

**Q1 — Why does MSE fail for multimodal registration?**  
In Part 2 you saw that the MSE landscape does *not* reach its minimum at the ground-truth alignment between T1 and T2 images, even though the anatomy is identical.  
- Explain in your own words why the same tissue (e.g. White Matter) has different intensities in T1 vs T2 MRI.
- What property of Mutual Information makes it insensitive to this contrast difference?
- Sketch (or describe) what the joint histogram looks like at perfect alignment vs. misalignment, and explain how that shape relates to the MI value.

---

**Q2 — Rigid vs. Affine vs. Deformable — when would you choose each?**  
Imagine three clinical scenarios:  
*(a)* Aligning a patient's follow-up brain MRI to their baseline scan from six months ago.  
*(b)* Aligning a CT scan of the abdomen taken during a breath-hold to an MRI taken freely breathing.  
*(c)* Aligning a T1 MRI from Scanner A (3T Siemens) to a T2 MRI from Scanner B (1.5T GE) in the same session.  
For each scenario, justify which transform family (rigid, affine, or deformable/B-spline) is most appropriate, and identify what physical phenomenon drives the choice.

---

### 🔍 Similarity Metrics & Optimization

**Q3 — NMI vs. Mattes MI: what is the practical difference?**  
SimpleITK's `SetMetricAsMattesMutualInformation` and `SetMetricAsJointHistogramMutualInformation` both estimate MI, but Mattes MI uses *random sampling* of 15 % of voxels per iteration.  
- What is the computational benefit of this sampling strategy?
- What risk does it introduce, and how does the multi-resolution pyramid mitigate that risk?
- Under what image conditions might you prefer the full joint histogram approach instead?

---

**Q4 — Multi-resolution registration and local minima.**  
The coarse-to-fine pyramid strategy (shrink factors `[4, 2, 1]`, sigmas `[2, 1, 0]`) is a core component of classical registration.  
- Explain intuitively why starting at 1/4 resolution helps the optimizer avoid local minima that would trap a full-resolution search.
- If you reduced the pyramid to a single level (full resolution only), what kinds of misalignments would be hardest to recover from? Why?
- How does the VoxelMorph U-Net architecture implicitly replicate this multi-scale reasoning through its encoder–decoder structure?

---

### 🧠 Deep Learning Registration

**Q5 — The role of the smoothness regularizer (`λ`).**  
The total loss is `L = L_LNCC + λ · L_smooth`. In the notebook `λ = 1.5`.  
- What physical constraint does `L_smooth` enforce on the displacement field?
- What would you expect to happen to the displacement field and the registered images if you set `λ = 0`? What about `λ = 100`?
- The Jacobian determinant diagnostic measures *folding* in the deformation. Explain what a negative Jacobian determinant means geometrically and why it is clinically unacceptable.
- How would you choose an appropriate `λ` in practice, given you have segmentation masks available?

---

**Q6 — Local NCC as a differentiable surrogate for NMI.**  
The `LocalNCC` loss uses a sliding window approach to measure local correlation rather than global intensity statistics.  
- Why is global NMI difficult to use directly as a training loss for a neural network?
- What does Local NCC capture that global (image-wide) NCC would miss in the presence of intensity inhomogeneities or partial-field images?
- The window size `win=9` (a 19×19 neighbourhood) was chosen in this tutorial. Describe the trade-off between a very small window (e.g. `win=2`) and a very large window (e.g. `win=40`).

---

### 📊 Evaluation & Clinical Validity

**Q7 — Why is NMI alone an insufficient evaluation metric?**  
Part 5 showed that registration can improve NMI while anatomical accuracy (Dice) remains poor — or vice versa.  
- Give a concrete example of a failure mode where NMI improves but the anatomy is actually *more* misaligned.
- Explain what information Dice Score, TRE (Target Registration Error), and the Jacobian determinant each add that NMI cannot provide.
- For a neurosurgery planning system, rank these four metrics in order of clinical importance and justify your ranking.

---

**Q8 — Speed vs. accuracy trade-offs in clinical deployment.**  
Part 6 benchmarked classical rigid registration (~hundreds of ms per pair) against DL inference (~single-digit ms per pair on GPU).  
- Identify two clinical workflows where inference speed is the dominant constraint and DL registration would be strongly preferred.
- Identify two scenarios where the iterative per-image-pair optimization of classical methods is preferable despite being slower.
- The DL model in this tutorial was trained on a *single phantom*. What are the likely failure modes when this model is applied to real MRI data from multiple scanners? What training-data strategy would you adopt to improve robustness?

---

### 🛠 Design & Extension

**Q9 — Extending to 3-D and diffeomorphic registration.**  
The tutorial operates on 2-D synthetic images for clarity, but clinical registration is almost always 3-D and should ideally be *diffeomorphic* (invertible and topology-preserving).  
- List the key code changes needed to adapt the `VoxelMorphUNet` and `SpatialTransformer` to 3-D volumes (consider layer types, memory, and compute).
- The *scaling-and-squaring* algorithm converts a stationary velocity field (SVF) into a diffeomorphic transformation. Why is guaranteeing diffeomorphism important for tasks like atlas construction or longitudinal change detection?
- What is the relationship between ANTs SyN (mentioned in the Further Reading) and the gradient-descent approach in this tutorial? What makes SyN *symmetric*?

---

**Q10 — Open-ended design challenge.**  
You are asked to build a production pipeline to automatically co-register T1 and FLAIR MRI scans for 10,000 patients across 5 hospital sites with different MRI vendors.  
Outline your design, addressing the following:  
1. **Preprocessing**: What steps would you apply before registration, and why?  
2. **Method choice**: Would you use classical, deep learning, or a hybrid approach? Justify.  
3. **Quality control**: How would you automatically flag failed registrations without manual inspection of all 10,000 cases?  
4. **Failure recovery**: What fallback strategy would you implement when automatic QC flags a case?  
5. **Validation**: What held-out test set and metrics would you use to sign off the pipeline before clinical deployment?
